# Задание

Решите задачу с данными train_small.csv, test_small.csv с помощью CatBoost, xgboost и LightGBM. Найдите оптимальные параметры. Сравните скорость обучения и качество. 

In [1]:
import pandas as pd
import time

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import roc_auc_score

from catboost import CatBoostClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

In [2]:
X_train = pd.read_csv('train_small.csv')
X_test = pd.read_csv('test_small.csv')

ytrain = X_train['Disbursed']
Xtrain = X_train.copy()
del Xtrain['Disbursed']

ytest = X_test['Disbursed']
Xtest = X_test.copy()
del Xtest['Disbursed']

In [3]:
Xtrain.columns = [f'feature_{i}' for i in range(Xtrain.shape[1])]
Xtest.columns = [f'feature_{i}' for i in range(Xtest.shape[1])]

# потому что вылезла ошибка LightGBMError: Do not support special JSON characters in feature name.

In [4]:
cat_params = {
    'depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'iterations': [100, 200]
}

# silent обучение, без early stopping внутри гридсерча

cat = CatBoostClassifier(
    loss_function='Logloss',
    eval_metric='AUC',
    verbose=0,
    random_seed=42
)

grid_cat = GridSearchCV(
    estimator=cat,
    param_grid=cat_params,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

start_time = time.time()
grid_cat.fit(Xtrain, ytrain)
end_time = time.time()

y_pred_cat = grid_cat.best_estimator_.predict_proba(Xtest)[:, 1]
auc_cat = roc_auc_score(ytest, y_pred_cat)

print("лучшие параметры catboost:", grid_cat.best_params_)
print("roc auc на тесте:", auc_cat)
print("время обучения (сек):", round(end_time - start_time, 2))

лучшие параметры catboost: {'depth': 4, 'iterations': 200, 'learning_rate': 0.05}
roc auc на тесте: 0.8397733601868307
время обучения (сек): 18.91


In [5]:
xgb_params = {
    'max_depth': [4, 6],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 200]
}

xgb = XGBClassifier(
    objective='binary:logistic',
    use_label_encoder=False,
    eval_metric='auc',
    random_state=42
)

grid_xgb = GridSearchCV(
    estimator=xgb,
    param_grid=xgb_params,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

start_time = time.time()
grid_xgb.fit(Xtrain, ytrain)
end_time = time.time()

y_pred_xgb = grid_xgb.best_estimator_.predict_proba(Xtest)[:, 1]
auc_xgb = roc_auc_score(ytest, y_pred_xgb)

print("лучшие параметры xgboost:", grid_xgb.best_params_)
print("roc auc на тесте:", auc_xgb)
print("время обучения (сек):", round(end_time - start_time, 2))


c:\Users\Anastasia\AppData\Local\Programs\Python\Python311\Lib\site-packages\xgboost\training.py:183: UserWarning: [23:55:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


лучшие параметры xgboost: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 200}
roc auc на тесте: 0.8256601719703129
время обучения (сек): 2.34


In [6]:
lgbm_params = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [4, 6],
    'n_estimators': [100, 200]
}

lgbm = LGBMClassifier(
    random_state=42
)

grid_lgbm = GridSearchCV(
    estimator=lgbm,
    param_grid=lgbm_params,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

start_time = time.time()
grid_lgbm.fit(Xtrain, ytrain)
end_time = time.time()

y_pred_lgbm = grid_lgbm.best_estimator_.predict_proba(Xtest)[:, 1]
auc_lgbm = roc_auc_score(ytest, y_pred_lgbm)

print("лучшие параметры лгбт или как там его:", grid_lgbm.best_params_)
print("roc auc на тесте:", auc_lgbm)
print("время обучения (сек):", round(end_time - start_time, 2))


лучшие параметры лгбт или как там его: {'learning_rate': 0.05, 'max_depth': 4, 'n_estimators': 100}
roc auc на тесте: 0.8325603086138593
время обучения (сек): 5.02


catboost дает лучшее качество, но учится дольше всех. его лучшие параметры: depth=4, lr=0.05, iters=200. xgboost учится быстрее всех, но его auc меньше всех. лучшие параметры: max_depth=4, lr=0.05, n_estim=200. lightgbm и сравнительно точный, и сравнительно быстрый. его параметры: max_depth=4, lr=0.05, n_estim=100

# Задание

Решите задачу классификации пассажиров Титаника с помощью любого из ансамблей (или нескольких из них). Если у вас есть наработки по Титанику, используйте их. 

Удалось ли улучшить качество с помощью какого-либо из этих алгоритмов?

In [7]:
from sklearn.metrics import accuracy_score

In [8]:
df = pd.read_csv("titanic.csv")

df['Title'] = df['Name'].str.extract(r',\s*([^\.]+)\.', expand=False)
df['Title'] = df['Title'].replace({
    'Mlle': 'Miss', 'Ms': 'Miss', 'Mme': 'Mrs',
    'Dr': 'Rare', 'Rev': 'Rare', 'Major': 'Rare', 'Col': 'Rare',
    'Capt': 'Rare', 'Countess': 'Rare', 'Sir': 'Rare',
    'Jonkheer': 'Rare', 'Lady': 'Rare', 'Don': 'Rare'
})

# размер семьи
df['FamilySize'] = df['SibSp'] + df['Parch'] + 1

# один
df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

# embarked
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

# age по title
df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

# дроп
df.drop(columns=['PassengerId', 'Name', 'Ticket', 'Cabin'], inplace=True)

df = pd.get_dummies(df, columns=['Sex', 'Embarked', 'Title'], drop_first=True)

X = df.drop(columns='Survived')
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

X_train.shape, X_test.shape, y_train.value_counts(), y_test.value_counts()


C:\Users\Anastasia\AppData\Local\Temp\ipykernel_28936\1297991690.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)


((668, 15),
 (223, 15),
 Survived
 0    412
 1    256
 Name: count, dtype: int64,
 Survived
 0    137
 1     86
 Name: count, dtype: int64)

In [9]:
param_grid = {
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 4, 6],
    'n_estimators': [100, 200]
}

lgbm = LGBMClassifier(random_state=42)

grid = GridSearchCV(
    estimator=lgbm,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1
)

start = time.time()
grid.fit(X_train, y_train)
end = time.time()

y_pred = grid.best_estimator_.predict(X_test)
y_proba = grid.best_estimator_.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
train_time = round(end - start, 2)

grid.best_params_, acc, auc, train_time


({'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 100},
 0.8116591928251121,
 np.float64(0.8636479375318282),
 1.94)

In [10]:
cat = CatBoostClassifier(
    iterations=100,
    learning_rate=0.05,
    depth=3,
    eval_metric='AUC',
    verbose=0,
    random_seed=42
)

start = time.time()
cat.fit(X_train, y_train)
end = time.time()

y_pred = cat.predict(X_test)
y_proba = cat.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_proba)
train_time = round(end - start, 2)

print("accuracy:", acc)
print("roc auc:", auc)
print("время:", train_time)


accuracy: 0.7937219730941704
roc auc: 0.8584281106773044
время: 0.1


модель lightgbm показала наилучшее качество на тесте, но требует больше времени на обучение. catboost работает быстрее и даёт сравнимо высокое качество, удобно без ручной обработки категориальных признаков. расширение фич и подбор параметров улучшили результат по сравнению с базовой моделью 